# Transformer encoder simple en Keras para análisis de sentimiento

## Sesión 6: Transformers / LLMs

En este notebook construiremos una versión simplificada de un **transformer encoder** usando Keras/TensorFlow.

La idea es entender cómo se combinan algunos componentes fundamentales de esta arquitectura:

- embeddings de tokens,
- embeddings de posición,
- self-attention,
- conexiones residuales,
- normalización,
- red feed-forward.

Aplicaremos este modelo a una tarea de **clasificación de sentimiento** usando reseñas del dominio **Amazon Baby Products**.

## Objetivo pedagógico

Al finalizar este notebook deberíamos poder:

- entender la estructura general de un encoder transformer,
- identificar las partes principales del modelo,
- construir un clasificador de texto con Keras/TensorFlow,
- relacionar esta arquitectura con lo visto antes en LSTM y con la idea de self-attention.

Este notebook no usa un modelo preentrenado.
Aquí el objetivo es **entender la construcción del encoder transformer**.

## Hoja de ruta del notebook

Seguiremos estos pasos:

1. cargar y preparar datos,
2. tokenizar y convertir el texto a secuencias,
3. definir embeddings de token y posición,
4. construir un bloque encoder transformer,
5. armar el modelo completo,
6. entrenar y evaluar,
7. comparar conceptualmente con LSTM.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from pandas import read_csv

from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split

np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)

## Función auxiliar para visualizar entrenamiento

Usaremos una función simple para graficar la evolución de accuracy y loss durante el entrenamiento.

In [2]:
def plot_history(history):
    history_df = pd.DataFrame(history.history)

    plt.figure(figsize=(6, 4))
    plt.plot(history_df["accuracy"], label="train_accuracy")
    if "val_accuracy" in history_df.columns:
        plt.plot(history_df["val_accuracy"], label="val_accuracy")
    plt.xlabel("Época")
    plt.ylabel("Accuracy")
    plt.title("Accuracy durante el entrenamiento")
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(6, 4))
    plt.plot(history_df["loss"], label="train_loss")
    if "val_loss" in history_df.columns:
        plt.plot(history_df["val_loss"], label="val_loss")
    plt.xlabel("Época")
    plt.ylabel("Loss")
    plt.title("Loss durante el entrenamiento")
    plt.legend()
    plt.tight_layout()
    plt.show()

## Cargamos el dataset

Trabajaremos con reseñas de productos del dominio **Amazon Baby Products**.

La tarea será de **clasificación binaria**:

- una reseña positiva,
- o una reseña negativa.

En este notebook asumimos que ya tenemos un archivo con dos columnas principales:

- `review`
- `label`

Si el nombre de tus columnas en el archivo original es distinto, solo habría que ajustarlo en la celda correspondiente.

In [ ]:
## Downloading data
!wget https://github.com/adoc-box/Datasets/blob/6d80f44a12aca282b7f2ff638866edef4806a282/amazon_baby.zip?raw=True

## Rename file
!mv "amazon_baby.zip?raw=True" "amazon_baby.zip"

## Un-compress downloaded zip file
!unzip amazon_baby.zip

In [ ]:
df = pd.read_csv('amazon_baby.csv')

df.head()

In [ ]:
## Lets assing label as: ratings {1,2} to 0, and the rest to 1
df['label'] = df['rating'].apply(lambda x: 0 if x in [1, 2] else 1)

df.head()

## Revisamos valores faltantes y distribución de clases

In [ ]:
df = df[["review", "label"]].dropna().copy()

print("Número de ejemplos:", len(df))
print("\nDistribución de clases:")
print(df["label"].value_counts())

## Miramos algunos ejemplos del texto

In [ ]:
for i in range(3):
    print(f"Ejemplo {i+1}")
    print("Label:", df.iloc[i]["label"])
    print("Review:", df.iloc[i]["review"][:500], "...")
    print("-" * 80)

## Separamos entrenamiento y test

Como en notebooks anteriores, separaremos los datos para poder evaluar generalización.

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["review"].values,
    df["label"].values,
    test_size=0.2,
    random_state=42,
    stratify=df["label"].values
)

print("Train:", len(X_train_text))
print("Test:", len(X_test_text))

## Tokenización y secuencias

El modelo no trabaja directamente con texto crudo.

Primero necesitamos:

1. construir un vocabulario,
2. convertir cada reseña en una secuencia de enteros,
3. hacer padding para que todas las secuencias tengan el mismo largo.

In [9]:
VOCAB_SIZE = 20000
MAX_LEN = 120

tokenizer = keras.preprocessing.text.Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

In [ ]:
## Display word-index in tokenizer
word_index = tokenizer.word_index
word_index

In [11]:
X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

In [ ]:
X_train_pad = keras.preprocessing.sequence.pad_sequences(
    X_train_seq, maxlen=MAX_LEN, padding="post", truncating="post"
)

X_test_pad = keras.preprocessing.sequence.pad_sequences(
    X_test_seq, maxlen=MAX_LEN, padding="post", truncating="post"
)

print("Shape train:", X_train_pad.shape)
print("Shape test :", X_test_pad.shape)

## ¿Qué hicimos aquí?

Cada reseña quedó representada como una secuencia de enteros.

Luego usamos **padding** para que todas las secuencias tengan la misma longitud máxima.

Esto es importante porque las redes neuronales suelen trabajar más cómodamente con tensores de tamaño fijo.

In [ ]:
print("Primer texto original:")
print(X_train_text[0][:300], "...")

print("\nPrimera secuencia:")
print(X_train_seq[0])

print("\nPrimera secuencia con padding:")
print(X_train_pad[0])

## Paso 1 del modelo: embeddings de token y posición

En un transformer, cada token se representa mediante un embedding.

Pero además necesitamos incorporar información de **posición**, porque el modelo no procesa la secuencia paso a paso como una RNN.

Por eso construiremos una capa que sume:

- embedding del token,
- embedding de posición.

In [16]:
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb = layers.Embedding(input_dim=maxlen, output_dim=embed_dim)

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

## ¿Qué hace esta capa?

Esta capa recibe una secuencia de enteros y devuelve una representación vectorial para cada posición.

La idea es:

- cada palabra se convierte en un vector,
- cada posición también se convierte en un vector,
- ambos se suman para que el modelo sepa **qué token** aparece y **dónde aparece**.

## Paso 2 del modelo: bloque encoder transformer

Ahora construiremos la pieza central del modelo.

Un bloque encoder simple contiene:

1. **self-attention**,
2. **conexión residual + normalización**,
3. **red feed-forward**,
4. **otra conexión residual + normalización**.

Esta será la parte que permite que el modelo relacione palabras entre sí usando atención.

In [17]:
class TransformerEncoderBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()

        key_dim = embed_dim // num_heads

        self.att = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=key_dim
        )

        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim),
        ])

        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)

        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, inputs, training=None):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)

        return self.layernorm2(out1 + ffn_output)

## ¿Qué está pasando dentro del bloque?

### 1. Self-attention
La capa `MultiHeadAttention` permite que cada posición de la secuencia atienda a otras posiciones.

### 2. Residual connection
Se suma la entrada original con la salida de atención.

### 3. Layer normalization
La normalización ayuda a estabilizar el entrenamiento.

### 4. Feed-forward network
Después de la atención, cada posición pasa por una pequeña red densa.

### 5. Segunda residual + normalización
Se repite la misma idea para cerrar el bloque.

## Definimos algunos hiperparámetros del modelo

In [18]:
EMBED_DIM = 32
NUM_HEADS = 4
FF_DIM = 64
DROPOUT_RATE = 0.1

## Construimos la arquitectura completa

La arquitectura será:

1. secuencia de enteros de entrada,
2. token + position embedding,
3. transformer encoder block,
4. `GlobalAveragePooling1D` para resumir la secuencia,
5. capas densas finales para clasificación binaria.

In [19]:
inputs = layers.Input(shape=(MAX_LEN,))

embedding_layer = TokenAndPositionEmbedding(MAX_LEN, VOCAB_SIZE, EMBED_DIM)
x = embedding_layer(inputs)

transformer_block = TransformerEncoderBlock(EMBED_DIM, NUM_HEADS, FF_DIM, DROPOUT_RATE)
x = transformer_block(x)

x = layers.GlobalAveragePooling1D()(x)
x = layers.Dropout(0.2)(x)
x = layers.Dense(32, activation="relu")(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs=inputs, outputs=outputs)

## ¿Por qué usamos `GlobalAveragePooling1D`?

Después del bloque encoder todavía tenemos una representación por posición.

Para clasificar toda la reseña, necesitamos resumir la secuencia completa en un solo vector.

Aquí usamos `GlobalAveragePooling1D`, que promedia las representaciones a lo largo de la secuencia.

In [ ]:
model.summary()

## Compilamos el modelo

Como es una tarea binaria, usaremos:

- `binary_crossentropy` como función de pérdida,
- `accuracy` como métrica.

In [21]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

## Entrenamos el modelo

Haremos un entrenamiento corto para que el notebook sea razonable para clase.

In [ ]:
history = model.fit(
    X_train_pad,
    y_train,
    validation_split=0.2,
    epochs=3,
    batch_size=32
)

## Visualizamos el entrenamiento

In [ ]:
plot_history(history)

## Evaluamos en test

In [ ]:
test_loss, test_acc = model.evaluate(X_test_pad, y_test)

print("Test loss:", test_loss)
print("Test accuracy:", test_acc)

## Probamos algunas predicciones nuevas

In [ ]:
sample_reviews = [
    "This baby product is amazing. It is easy to use and very helpful.",
    "I am very disappointed. The quality is poor and it broke quickly.",
    "The product is okay, but I expected something better for the price."
]

sample_seq = tokenizer.texts_to_sequences(sample_reviews)
sample_pad = keras.preprocessing.sequence.pad_sequences(
    sample_seq, maxlen=MAX_LEN, padding="post", truncating="post"
)

pred_probs = model.predict(sample_pad)
pred_labels = (pred_probs > 0.5).astype(int).flatten()

for review, prob, label in zip(sample_reviews, pred_probs.flatten(), pred_labels):
    print("Review:", review)
    print("Probabilidad positiva:", float(prob))
    print("Predicción:", "positiva" if label == 1 else "negativa")
    print("-" * 80)

## Comparación conceptual con la sesión 4

En la sesión 4 usamos un enfoque recurrente, por ejemplo:

**Embedding + LSTM + Dense**

Aquí usamos:

**Token + Position Embedding + Transformer Encoder + Dense**

La diferencia conceptual central es:

- en LSTM el contexto se modela de manera recurrente,
- en el transformer encoder el contexto se modela mediante **self-attention**.

## Ideas importantes para recordar

- Este notebook no usa un transformer preentrenado.
- Aquí construimos una versión simple de un **encoder transformer**.
- La atención permite relacionar distintas posiciones de la secuencia.
- Los embeddings de posición son importantes porque el orden sigue importando.
- Esta arquitectura puede usarse para clasificación de texto.

## Cierre

En este notebook vimos cómo construir un transformer encoder simple en Keras/TensorFlow para análisis de sentimiento.

El flujo general fue:

1. tokenizar texto,
2. agregar embeddings de token y posición,
3. aplicar un bloque encoder transformer,
4. resumir la secuencia,
5. clasificar la reseña.

Con esto ya tenemos una visión más concreta de cómo se arma internamente un modelo de este tipo.